# 05 — Hyperparameter Tuning IndoBERT (Lengkap)

Notebook ini melakukan HPO untuk fine-tuning IndoBERT dengan protokol yang
sama seperti BiLSTM:

- tuning hanya di train + validation,
- holdout hanya diuji sekali,
- semua trial dicatat,
- model terbaik di-retrain lalu diuji di holdout.

Ruang pencarian mencakup model checkpoint, panjang token, learning rate,
batch, weight decay, warmup, scheduler, optimizer, gradient clipping,
gradient accumulation, freeze layer, dropout, class weight, label smoothing,
presisi (fp32/fp16/bf16), dan metrik seleksi.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebook biasanya dijalankan dari folder notebooks/. Kita pindah ke root
# project agar semua path artefak konsisten.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "artifacts" / "data"
BERT_DIR = PROJECT_ROOT / "artifacts" / "bert" / "input"
RUNS_DIR = PROJECT_ROOT / "artifacts" / "notebook_runs"
for path in (DATA_DIR, BERT_DIR, RUNS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Data dir     :", DATA_DIR)
print("BERT input   :", BERT_DIR)
print("Runs dir     :", RUNS_DIR)

In [ ]:
from app.experiment.configs import DataConfig
from app.experiment.data_pipeline import prepare_data

config_path = DATA_DIR / "data_config.json"
if config_path.exists():
    data_config = DataConfig.from_dict(json.loads(config_path.read_text(encoding="utf-8")))
else:
    candidates = [
        PROJECT_ROOT / "ocr_output" / "data-mengandung.csv",
        PROJECT_ROOT.parent / "old" / "ocr_output" / "data-mengandung.csv",
        PROJECT_ROOT / "data.csv",
    ]
    csv_path = next((path for path in candidates if path.exists()), None)
    if csv_path is None:
        raise FileNotFoundError(
            "CSV tidak ditemukan. Set DataConfig.csv_input secara manual "
            "ke file dengan kolom: nama_produk;text;label"
        )
    data_config = DataConfig(
        csv_input=str(csv_path),
        output_dir=str(DATA_DIR),
        bert_input_dir=str(BERT_DIR),
        sort_source=True,
    )

# prepare_data bersifat deterministik dan cepat; aman dijalankan ulang.
bundle = prepare_data(data_config)
print(bundle.summary_frame().to_string(index=False))

In [ ]:
from app.experiment.configs import BertConfig, HPOConfig
from app.experiment.search_spaces import bert_search_space, preset_trials, space_summary
from app.experiment.hpo import HPOEngine
from app.experiment.bert_pipeline import run_bert_trial, train_final_bert

PRESET = "smoke"       # ganti ke "standard" atau "full" untuk paper
N_TRIALS = preset_trials(PRESET)
space = bert_search_space(PRESET)
print(f"Preset={PRESET}, target trial={N_TRIALS}, jumlah parameter={len(space.params)}")
pd.DataFrame(space_summary(space))

## Objective function

Setiap trial memuat tokenizer + model dari checkpoint, melatih, lalu
mengembalikan metrik validation. Model tidak disimpan per trial agar disk
tidak penuh; hanya model terbaik yang disimpan ulang di tahap akhir.

In [ ]:
def objective(params, trial_index, trial_dir):
    config = BertConfig.from_dict(params)
    result = run_bert_trial(
        bundle,
        config,
        trial_dir,
        save_model=False,
        save_predictions=False,
        evaluate_holdout=False,
        verbose=False,
    )
    return {"metrics": result.metrics, "artifacts": result.artifacts}

## Jalankan HPO

Untuk GPU 16 GB, mulai dari `smoke`. Setelah pipeline terbukti jalan, naikkan
ke `standard`. Untuk `full`, pertimbangkan menjalankan di HPC dan menyimpan
`run_dir` agar bisa di-resume.

In [ ]:
HPO_DIR = RUNS_DIR / "bert_hpo"
hpo_config = HPOConfig(
    strategy="random",
    n_trials=N_TRIALS,
    seed=42,
    timeout_minutes=240,
    objective_metric="val_f1",
    objective_mode="max",
    output_dir=str(HPO_DIR),
    study_name=f"bert_{PRESET}",
    resume=True,
    save_trial_models=False,
)

engine = HPOEngine(hpo_config, space, run_dir=HPO_DIR / f"{PRESET}_seed42", verbose=True)
hpo_result = engine.run(objective)
print("Run dir:", hpo_result.run_dir)
if hpo_result.best is None:
    raise RuntimeError("Tidak ada trial sukses. Cek error di trials.jsonl.")

In [ ]:
leaderboard = pd.read_csv(hpo_result.run_dir / "leaderboard.csv")
columns = ["trial_id", "status", "objective", "metric_val_f1", "metric_val_roc_auc", "metric_val_loss", "duration_sec"]
columns = [column for column in columns if column in leaderboard.columns]
leaderboard[columns].head(15)

## Plot progres dan importance

Grafik ini membantu menjawab: apakah pencarian membaik, dan parameter mana
yang paling berpengaruh? Sertakan interpretasinya di pembahasan paper.

In [ ]:
from app.experiment.reporting import plot_hpo_progress, plot_param_importance

plot_hpo_progress(
    leaderboard,
    hpo_result.run_dir / "hpo_progress.png",
    mode=hpo_config.objective_mode,
)
plot_param_importance(leaderboard, hpo_result.run_dir / "hpo_importance.png")
print("Plot tersimpan di", hpo_result.run_dir)

## Retrain model terbaik dan uji holdout

Setelah konfigurasi terbaik dibekukan dari validation, latih ulang dan
evaluasi holdout satu kali.

In [ ]:
best = hpo_result.best
best_config = BertConfig.from_dict(best.params)
BEST_OUT = RUNS_DIR / "bert_tuned"

best_output = train_final_bert(
    bundle,
    best_config,
    output_dir=BEST_OUT,
    hpo_info={
        "source": str(hpo_result.run_dir),
        "best_trial": best.trial_id,
        "best_params": best.params,
        "best_metrics": best.metrics,
        "search_space": space.to_dict(),
        "n_trials": len(hpo_result.trials),
    },
    verbose=True,
)
print(json.dumps(best_output["metrics"]["validation"], indent=2))
print(json.dumps(best_output["metrics"]["holdout"], indent=2))

## Bandingkan baseline vs tuned

Notebook 04 harus sudah dijalankan agar perbandingan tersedia.

In [ ]:
from app.experiment.reporting import compare_two_models, load_probabilities

baseline_path = RUNS_DIR / "bert_baseline" / "probs_holdout.csv"
tuned_path = BEST_OUT / "probs_holdout.csv"
if baseline_path.exists() and tuned_path.exists():
    y_base, p_base, _ = load_probabilities(baseline_path)
    y_tuned, p_tuned, _ = load_probabilities(tuned_path)
    comparison = compare_two_models(
        y_base, p_base, p_tuned,
        name_a="IndoBERT baseline", name_b="IndoBERT tuned",
        n_bootstrap=2000, seed=42,
    )
    print(json.dumps(comparison, indent=2, ensure_ascii=False))
else:
    print("Jalankan notebook 04 dulu untuk perbandingan baseline.")

## Yang perlu dilaporkan untuk HPO BERT

1. Checkpoint/model yang dicoba.
2. Rentang learning rate, batch, epoch, max_len.
3. Regularisasi: weight decay, dropout, label smoothing.
4. Strategi freeze layer.
5. Presisi (fp32/fp16/bf16) dan hardware.
6. Jumlah trial + strategi.
7. Metrik seleksi dan protokol holdout.
8. Uji statistik vs baseline.

Untuk paper, jalankan minimal 2–3 seed pada konfigurasi terbaik dan laporkan
mean ± std. Jangan memilih model berdasarkan holdout.